# 03 — Degradation Analysis

Load saved Phase 3 results and analyze how def-use probe accuracy degrades
with token distance and context length.

**Prerequisite:** run `scripts/run_experiment.py --phase 3` to generate results.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.analysis.metrics import degradation_slope, summary_table
from src.analysis.visualization import plot_degradation_heatmap, plot_layer_curves

RESULTS_DIR = Path('../results/')
MODEL_NAME = 'deepseek-coder-1.3b'

## 1. Load Phase 3 degradation results

In [2]:
csv_path = RESULTS_DIR / f'phase3_{MODEL_NAME}' / 'phase3_degradation.csv'

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} records")
    print(df.head())
else:
    print(f"Results not found at {csv_path}")
    print("Run: python scripts/run_experiment.py --phase 3 --model deepseek-coder-1.3b")
    df = None

Results not found at ../results/phase3_deepseek-coder-1.3b/phase3_degradation.csv
Run: python scripts/run_experiment.py --phase 3 --model deepseek-coder-1.3b


## 2. Accuracy vs filler size (all layers)

In [3]:
if df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    
    for ax, ftype in zip(axes, ['irrelevant', 'lexical_decoy']):
        sub = df[df['filler_type'] == ftype]
        agg = sub.groupby(['filler_tokens', 'layer'])['accuracy'].mean().reset_index()
        
        for layer, grp in agg.groupby('layer'):
            grp = grp.sort_values('filler_tokens')
            ax.plot(grp['filler_tokens'], grp['accuracy'], marker='o',
                    label=f'L{layer}', alpha=0.8, linewidth=1.5)
        
        ax.set_xlabel('Filler tokens inserted')
        ax.set_ylabel('Def-use edge probe accuracy')
        ax.set_title(f'Filler type: {ftype}')
        ax.legend(title='Layer', fontsize=7, ncol=2, framealpha=0.7)
        sns.despine(ax=ax)
    
    fig.suptitle(f'Phase 3: Degradation with Context Distance — {MODEL_NAME}', fontsize=13)
    plt.tight_layout()
    plt.show()

## 3. Degradation slope analysis

In [4]:
if df is not None:
    print("Degradation slopes (accuracy vs layer, per filler type):")
    slopes = degradation_slope(df.rename(columns={'filler_type': 'distance_bucket'}),
                               metric='accuracy', group_col='distance_bucket')
    for k, v in slopes.items():
        print(f"  {k}: {v:+.4f}")

## 4. Heatmap: irrelevant filler

In [5]:
if df is not None:
    sub = df[df['filler_type'] == 'irrelevant'].copy()
    sub = sub.groupby(['filler_tokens', 'layer'])['accuracy'].mean().reset_index()
    sub.columns = ['distance_bucket', 'layer', 'accuracy']
    
    fig = plot_degradation_heatmap(
        sub, metric='accuracy',
        title=f'Def-Use Probe Accuracy — Irrelevant Filler ({MODEL_NAME})'
    )
    plt.show()

## 5. Lexical decoy effect

In [6]:
if df is not None:
    # Compare irrelevant vs lexical_decoy at each filler size
    pivot = df.groupby(['filler_type', 'filler_tokens'])['accuracy'].mean().unstack(0)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    for col in pivot.columns:
        ax.plot(pivot.index, pivot[col], marker='o', label=col)
    ax.set_xlabel('Filler tokens inserted')
    ax.set_ylabel('Mean def-use probe accuracy (all layers)')
    ax.set_title('Effect of filler type on probe accuracy')
    ax.legend()
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()
    
    print("\nKey finding: if lexical_decoy accuracy drops more than irrelevant,")
    print("the model is relying on surface name matching rather than semantic tracking.")